# Crash Course — Proteomics (단백질체)

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/proteomics.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/proteomics.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 최소 실행 예시

### 예시 1 — log 변환, 중앙값 정규화, 결측 패턴, 단백질별 검정 + BH

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(42)
n_prot, n_per = 300, 6
samples = [f"C{i}" for i in range(1, 7)] + [f"D{i}" for i in range(1, 7)]
group = np.array(["control"] * n_per + ["disease"] * n_per)

# 가상 데이터: 단백질마다 기본 풍부도(log2)가 다르고, 앞의 30개만 disease에서 2배(+1) 높다
base = rng.normal(24, 2, n_prot)
log_true = base[:, None] + rng.normal(0, 0.35, (n_prot, 12))
log_true[:30, n_per:] += 1.0
load = rng.normal(0, 0.4, 12)                       # 시료마다 주입량이 조금씩 다르다
raw = pd.DataFrame(2 ** (log_true + load), columns=samples,
                   index=[f"P{i:03d}" for i in range(n_prot)])

# 결측 심기: (1) 강도가 낮을수록 잘 빠진다(MNAR)  (2) 강도와 무관하게 2%가 빠진다(MCAR)
p_mnar = 1 / (1 + np.exp((np.log2(raw) - 21.5) * 2))
raw = raw.mask(rng.random(raw.shape) < p_mnar).mask(rng.random(raw.shape) < 0.02)

# 1) log2 변환 → 2) 중앙값 정규화(시료 중앙값을 전체 평균 중앙값에 맞춘다)
logx = np.log2(raw)
med = logx.median()
norm = logx - med + med.mean()
print("시료별 중앙값 (정규화 전):", med.round(2).tolist())
print("시료별 중앙값 (정규화 후):", norm.median().round(2).tolist())

# 3) 결측 패턴 확인
miss = norm.isna()
print(f"\n전체 결측 비율: {miss.to_numpy().mean():.1%}")
print("시료별 결측 비율:", miss.mean().round(2).tolist())
q = pd.qcut(norm.mean(axis=1), 4, labels=["Q1(낮음)", "Q2", "Q3", "Q4(높음)"])
print("평균 강도 사분위별 결측 비율:")
print(miss.mean(axis=1).groupby(q, observed=True).mean().round(3).to_string())

# 4) 군마다 유효값이 4개 이상인 단백질만 검정 (결측은 채우지 않고 빼고 계산)
ctrl, dis = norm.loc[:, group == "control"], norm.loc[:, group == "disease"]
keep = (ctrl.notna().sum(axis=1) >= 4) & (dis.notna().sum(axis=1) >= 4)
t, p = stats.ttest_ind(dis[keep], ctrl[keep], axis=1, equal_var=False, nan_policy="omit")
res = pd.DataFrame({"log2FC": dis[keep].mean(axis=1) - ctrl[keep].mean(axis=1), "p": np.asarray(p)})
res["padj"] = multipletests(res["p"], method="fdr_bh")[1]
sig = res[res["padj"] < 0.05]
truth = set(raw.index[:30])
print(f"\n검정한 단백질: {keep.sum()} / {n_prot}  (제외 {n_prot - keep.sum()}개는 유효값 부족)")
print(f"raw p < 0.05: {(res['p'] < 0.05).sum()}개,  BH padj < 0.05: {len(sig)}개")
print(f"padj < 0.05 중 진짜 변화: {len(set(sig.index) & truth)}개, 거짓 양성: {len(set(sig.index) - truth)}개")
print(sig.sort_values("padj").head(5).round(4).to_string())

# 자기 점검: 정규화 후 시료 중앙값이 모두 같아야 한다
assert np.allclose(norm.median(), norm.median().iloc[0])

### 예시 2 — 결측을 0으로 채우면

예시 1에 이어서 실행한다.

In [ ]:
zero = norm.fillna(0)                     # 흔한 실수: "안 보였으니 0"
fc0 = zero.loc[:, group == "disease"].mean(axis=1) - zero.loc[:, group == "control"].mean(axis=1)
p0 = stats.ttest_ind(zero.loc[:, group == "disease"], zero.loc[:, group == "control"], axis=1, equal_var=False)[1]
padj0 = multipletests(p0, method="fdr_bh")[1]
null = ~norm.index.isin(list(truth))      # 실제로는 변화가 없는 270개
print("\n[0으로 채운 경우] 변화 없는 단백질 270개 중")
print(f"  |log2FC| > 1 인 것: {(fc0[null].abs() > 1).sum()}개   (결측 제외 계산에서는 {(res.loc[res.index.isin(norm.index[null]), 'log2FC'].abs() > 1).sum()}개)")
print(f"  가장 큰 |log2FC|: {fc0[null].abs().max():.1f}  → 2^{fc0[null].abs().max():.1f}배 차이로 읽힌다")
print(f"  padj < 0.05: {(padj0[null] < 0.05).sum()}개")
worst = fc0[null].abs().idxmax()
print(f"  예: {worst} 의 값 →", norm.loc[worst].round(1).tolist())